In [15]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-41.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-41.xlsx',index=None)
data


转换完成
删除时间不是2022年的之后的数据： 342706
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2240380082.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2240380082.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 325213
325213
325213
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 286896
截断过长时间怠速的运行时间为：2433.0292541980743s
开始线性插值...
插值后数据量 303492
插值运行时间为：3.584911346435547s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,2801.0,108.941504,37.652724,5.0,2022-10-20 12:24:56,15731386.0,1.0,25.0,830.0,1.388889,0.555556
1,2801.0,108.941520,37.652732,7.0,2022-10-20 12:24:57,15731387.0,3.0,63.0,1105.0,1.944444,0.555556
2,2801.0,108.941520,37.652752,8.0,2022-10-20 12:24:58,15731388.0,5.0,97.0,1105.0,2.222222,0.277778
3,2801.0,108.941552,37.652764,8.0,2022-10-20 12:24:59,15731389.0,8.0,110.0,764.0,2.222222,0.000000
4,2801.0,108.941552,37.652772,9.0,2022-10-20 12:25:00,15731390.0,10.0,135.0,997.0,2.500000,0.277778
...,...,...,...,...,...,...,...,...,...,...,...
303487,3121.0,109.041919,37.748804,0.0,2022-10-30 13:22:22,16598832.0,42744.0,218417.0,663.0,0.000000,0.000000
303488,3121.0,109.041919,37.748804,0.0,2022-10-30 13:22:23,16598833.0,42744.0,218428.0,669.0,0.000000,0.000000
303489,3121.0,109.041919,37.748804,0.0,2022-10-30 13:22:24,16598834.0,42744.0,218440.0,670.0,0.000000,0.000000
303490,3121.0,109.041919,37.748804,0.0,2022-10-30 13:22:25,16598835.0,42744.0,218448.0,664.0,0.000000,0.000000


In [16]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-42.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-42.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 326757
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\868558609.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\868558609.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 315300
315300
315300
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 270667
截断过长时间怠速的运行时间为：2747.771801710129s
开始线性插值...
插值后数据量 280598
插值运行时间为：2.5395665168762207s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,3122.0,109.041159,37.748495,5.0,2022-10-30 13:25:42,16599032.0,115.0,1008.0,650.0,1.388889,-0.277778
1,3122.0,109.041143,37.748472,4.0,2022-10-30 13:25:43,16599033.0,116.0,1019.0,637.0,1.111111,-0.277778
2,3122.0,109.041128,37.748447,2.0,2022-10-30 13:25:44,16599034.0,117.0,1032.0,635.0,0.555556,-0.555556
3,3122.0,109.041128,37.748452,1.0,2022-10-30 13:25:45,16599035.0,118.0,1066.0,1015.0,0.277778,-0.277778
4,3122.0,109.041143,37.748416,2.0,2022-10-30 13:25:46,16599036.0,118.0,1128.0,1280.0,0.555556,0.277778
...,...,...,...,...,...,...,...,...,...,...,...
280593,3486.0,108.941032,37.647876,0.0,2022-11-11 04:06:59,17602309.0,1517.0,9732.0,298.0,0.000000,0.000000
280594,3486.0,108.941032,37.647876,0.0,2022-11-11 04:07:00,17602310.0,1517.0,9742.0,123.0,0.000000,0.000000
280595,3486.0,108.941032,37.647876,0.0,2022-11-11 04:07:01,17602311.0,1517.0,9744.0,30.0,0.000000,0.000000
280596,3486.0,108.941032,37.647876,0.0,2022-11-11 04:07:02,17602312.0,1517.0,9744.0,2.0,0.000000,0.000000


In [17]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-43.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-43.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 331268
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\206767818.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\206767818.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 318625
318625
318625
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 269224
截断过长时间怠速的运行时间为：3047.9140071868896s
开始线性插值...
插值后数据量 280129
插值运行时间为：2.511688470840454s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,3487.0,108.941127,37.647880,8.0,2022-11-11 04:10:45,17602535.0,3.0,40.0,1245.0,2.222222,0.555556
1,3487.0,108.941127,37.647839,10.0,2022-11-11 04:10:46,17602536.0,6.0,67.0,1398.0,2.777778,0.555556
2,3487.0,108.941127,37.647816,10.0,2022-11-11 04:10:47,17602537.0,9.0,79.0,1412.0,2.777778,0.000000
3,3487.0,108.941127,37.647788,9.0,2022-11-11 04:10:48,17602538.0,12.0,88.0,1294.0,2.500000,-0.277778
4,3487.0,108.941127,37.647768,9.0,2022-11-11 04:10:49,17602539.0,14.0,106.0,1247.0,2.500000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
280124,3929.0,108.942383,37.661215,0.0,2022-11-22 12:53:20,18584290.0,1072.0,7365.0,210.0,0.000000,0.000000
280125,3929.0,108.942383,37.661215,0.0,2022-11-22 12:53:21,18584291.0,1072.0,7372.0,61.0,0.000000,0.000000
280126,3929.0,108.942383,37.661215,0.0,2022-11-22 12:53:22,18584292.0,1072.0,7373.0,33.0,0.000000,0.000000
280127,3929.0,108.942383,37.661215,0.0,2022-11-22 12:53:23,18584293.0,1072.0,7373.0,0.0,0.000000,0.000000


In [18]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-51.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-51.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 332877
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2293024793.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2293024793.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 321791
321791
321791
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 251600
截断过长时间怠速的运行时间为：4208.147475004196s
开始线性插值...
插值后数据量 261214
插值运行时间为：2.2796175479888916s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,3930.0,108.941696,37.659484,0.0,2022-11-22 13:00:44,18584734.0,129.0,2015.0,637.0,0.0,0.0
1,3930.0,108.941735,37.659380,0.0,2022-11-22 13:00:45,18584735.0,129.0,2027.0,634.0,0.0,0.0
2,3930.0,108.941808,37.659372,0.0,2022-11-22 13:00:46,18584736.0,129.0,2039.0,643.0,0.0,0.0
3,3930.0,108.941808,37.659383,0.0,2022-11-22 13:00:47,18584737.0,129.0,2050.0,638.0,0.0,0.0
4,3930.0,108.941768,37.659396,0.0,2022-11-22 13:00:48,18584738.0,129.0,2054.0,642.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
261209,4297.0,108.938504,37.646864,0.0,2022-12-13 21:00:38,20427928.0,8987.0,72900.0,30.0,0.0,0.0
261210,4297.0,108.938504,37.646864,0.0,2022-12-13 21:00:39,20427929.0,8987.0,72900.0,20.0,0.0,0.0
261211,4297.0,108.938504,37.646864,0.0,2022-12-13 21:00:32,20427922.0,8987.0,72900.0,0.0,0.0,-0.0
261212,4298.0,108.938504,37.646812,0.0,2022-12-13 23:45:29,20437819.0,0.0,158.0,752.0,0.0,0.0


In [19]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-52.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-52.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 296059
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1231340311.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1231340311.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 285004
285004
285004
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 231073
截断过长时间怠速的运行时间为：2964.834979057312s
开始线性插值...
插值后数据量 240781
插值运行时间为：2.1671721935272217s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,4299.0,108.938656,37.646932,0.0,2022-12-14 02:57:05,20449315.0,0.0,687.0,829.0,0.0,0.0
1,4299.0,108.938656,37.646932,0.0,2022-12-14 02:57:06,20449316.0,0.0,711.0,827.0,0.0,0.0
2,4299.0,108.938672,37.646936,0.0,2022-12-14 02:57:07,20449317.0,0.0,735.0,829.0,0.0,0.0
3,4299.0,108.938656,37.646936,0.0,2022-12-14 02:57:08,20449318.0,0.0,759.0,827.0,0.0,0.0
4,4299.0,108.938656,37.646932,0.0,2022-12-14 02:57:09,20449319.0,0.0,784.0,828.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
240776,4552.0,109.276432,38.000223,0.0,2022-12-30 23:55:50,21907240.0,61770.0,179485.0,74.0,0.0,0.0
240777,4552.0,109.276432,38.000223,0.0,2022-12-30 23:55:51,21907241.0,61770.0,179486.0,36.0,0.0,0.0
240778,4552.0,109.276432,38.000223,0.0,2022-12-30 23:55:45,21907235.0,61770.0,179486.0,0.0,0.0,-0.0
240779,4552.0,109.276432,38.000223,0.0,2022-12-30 23:55:46,21907236.0,61770.0,179486.0,0.0,0.0,0.0


转换完成
删除时间不是2022年的之后的数据： 392303
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1702751838.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1702751838.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 376387
376387
376387
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 330357
截断过长时间怠速的运行时间为：3279.088973045349s
开始线性插值...
插值后数据量 345074
插值运行时间为：3.4798765182495117s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,950.0,108.941336,37.647452,0.0,2022-08-23 08:57:21,10707731.0,7.0,181.0,646.0,0.0,0.0
1,950.0,108.941336,37.647452,0.0,2022-08-23 08:57:22,10707732.0,7.0,192.0,649.0,0.0,0.0
2,950.0,108.941336,37.647452,0.0,2022-08-23 08:57:23,10707733.0,7.0,202.0,619.0,0.0,0.0
3,950.0,108.941336,37.647456,0.0,2022-08-23 08:57:24,10707734.0,7.0,214.0,634.0,0.0,0.0
4,950.0,108.941336,37.647459,0.0,2022-08-23 08:57:25,10707735.0,7.0,224.0,635.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
345069,1474.0,109.276544,38.023572,0.0,2022-09-10 19:32:44,12301054.0,1464.0,16104.0,106.0,0.0,0.0
345070,1474.0,109.276544,38.023572,0.0,2022-09-10 19:32:45,12301055.0,1464.0,16107.0,25.0,0.0,0.0
345071,1474.0,109.276544,38.023572,0.0,2022-09-10 19:32:46,12301056.0,1464.0,16107.0,0.0,0.0,0.0
345072,1474.0,109.276544,38.023572,0.0,2022-09-10 19:32:47,12301057.0,1464.0,16107.0,0.0,0.0,0.0


转换完成
删除时间不是2022年的之后的数据： 309741
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\543615090.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\543615090.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 301239
301239
301239
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 265439
截断过长时间怠速的运行时间为：2129.8745856285095s
开始线性插值...
插值后数据量 273679
插值运行时间为：2.1389870643615723s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,1475.0,109.276432,38.023551,1.0,2022-09-10 19:41:28,12301578.0,0.0,36.0,1210.0,0.277778,0.555556
1,1475.0,109.276432,38.023548,3.0,2022-09-10 19:41:29,12301579.0,1.0,76.0,749.0,0.833333,0.555556
2,1475.0,109.276479,38.023551,5.0,2022-09-10 19:41:30,12301580.0,3.0,111.0,716.0,1.388889,0.555556
3,1475.0,109.276512,38.023544,6.0,2022-09-10 19:41:31,12301581.0,4.0,152.0,878.0,1.666667,0.277778
4,1475.0,109.276528,38.023531,7.0,2022-09-10 19:41:32,12301582.0,6.0,202.0,1058.0,1.944444,0.277778
...,...,...,...,...,...,...,...,...,...,...,...
273674,1903.0,109.012424,37.641259,0.0,2022-09-25 02:43:49,13536519.0,314.0,3328.0,0.0,0.000000,-0.000000
273675,1903.0,109.012424,37.641259,0.0,2022-09-25 02:43:50,13536520.0,314.0,3328.0,0.0,0.000000,0.000000
273676,1905.0,109.012368,37.641256,0.0,2022-09-25 02:47:12,13536722.0,0.0,30.0,1266.0,0.000000,0.000000
273677,1906.0,109.011767,37.641236,2.0,2022-09-25 02:52:11,13537021.0,0.0,21.0,650.0,0.555556,0.000000


In [14]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL1202005070258-33.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-04-21 10:35:10', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL1202005070258-33.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 349831
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\701616102.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\701616102.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 336679
336679
336679
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 287722
截断过长时间怠速的运行时间为：3171.515713453293s
开始线性插值...
插值后数据量 299276
插值运行时间为：2.7120954990386963s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,2300.0,109.226559,38.246344,4.0,2022-10-07 21:22:18,14640428.0,1.0,13.0,674.0,1.111111,0.000000
1,2300.0,109.226543,38.246331,4.0,2022-10-07 21:22:19,14640429.0,2.0,28.0,677.0,1.111111,0.000000
2,2300.0,109.226543,38.246328,5.0,2022-10-07 21:22:20,14640430.0,4.0,43.0,671.0,1.388889,0.277778
3,2300.0,109.226559,38.246331,5.0,2022-10-07 21:22:21,14640431.0,5.0,58.0,671.0,1.388889,0.000000
4,2300.0,109.226592,38.246348,5.0,2022-10-07 21:22:22,14640432.0,7.0,73.0,683.0,1.388889,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
299271,2798.0,108.941424,37.652235,0.0,2022-10-20 12:04:17,15730147.0,722.0,8094.0,409.0,0.000000,0.000000
299272,2798.0,108.941424,37.652235,0.0,2022-10-20 12:04:18,15730148.0,722.0,8106.0,197.0,0.000000,0.000000
299273,2798.0,108.941424,37.652240,0.0,2022-10-20 12:04:19,15730149.0,722.0,8113.0,46.0,0.000000,0.000000
299274,2798.0,108.941424,37.652240,0.0,2022-10-20 12:04:20,15730150.0,722.0,8113.0,28.0,0.000000,0.000000


In [32]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-51.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-51.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 305606
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\731461131.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\731461131.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 293970
293970
293970
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 232852
截断过长时间怠速的运行时间为：3542.3008556365967s
开始线性插值...
插值后数据量 241774
插值运行时间为：1.8681261539459229s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,2400.0,108.941336,37.653852,25.0,2022-11-05 23:21:30,12489648.0,66.0,1110.0,1224.0,6.944444,0.000000
1,2400.0,108.941336,37.653915,25.0,2022-11-05 23:21:31,12489649.0,71.0,1192.0,1232.0,6.944444,0.000000
2,2400.0,108.941336,37.653976,25.0,2022-11-05 23:21:32,12489650.0,80.0,1258.0,1212.0,6.944444,0.000000
3,2400.0,108.941336,37.654032,25.0,2022-11-05 23:21:33,12489651.0,87.0,1261.0,1200.0,6.944444,0.000000
4,2400.0,108.941336,37.654095,24.0,2022-11-05 23:21:34,12489652.0,94.0,1261.0,1168.0,6.666667,-0.277778
...,...,...,...,...,...,...,...,...,...,...,...
241769,2776.0,109.498303,38.259840,0.0,2022-11-24 09:57:04,14082982.0,116217.0,665193.0,286.0,0.000000,0.000000
241770,2776.0,109.498303,38.259840,0.0,2022-11-24 09:57:05,14082983.0,116217.0,665211.0,69.0,0.000000,0.000000
241771,2776.0,109.498303,38.259840,0.0,2022-11-24 09:57:06,14082984.0,116217.0,665214.0,0.0,0.000000,0.000000
241772,2776.0,109.498303,38.259840,0.0,2022-11-24 09:57:07,14082985.0,116217.0,665214.0,0.0,0.000000,0.000000


In [33]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-52.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-52.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 409379
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\3327590437.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\3327590437.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 396786
396786
396786
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 280749
截断过长时间怠速的运行时间为：8018.93162727356s
开始线性插值...
插值后数据量 289919
插值运行时间为：2.3590822219848633s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,2777.0,109.498303,38.259848,0.0,2022-11-24 09:59:31,14083129.0,0.0,11.0,547.0,0.0,0.0
1,2777.0,109.498303,38.259848,0.0,2022-11-24 09:59:32,14083130.0,0.0,26.0,552.0,0.0,0.0
2,2777.0,109.498303,38.259848,0.0,2022-11-24 09:59:33,14083131.0,0.0,43.0,555.0,0.0,0.0
3,2777.0,109.498303,38.259848,0.0,2022-11-24 09:59:34,14083132.0,0.0,59.0,557.0,0.0,0.0
4,2777.0,109.498303,38.259848,0.0,2022-11-24 09:59:35,14083133.0,0.0,76.0,558.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
289914,3192.0,108.988639,37.645424,0.0,2022-12-07 03:28:27,15182865.0,1675.0,43197.0,0.0,0.0,0.0
289915,3192.0,108.988655,37.645412,0.0,2022-12-07 03:28:28,15182866.0,1675.0,43197.0,0.0,0.0,0.0
289916,3192.0,108.988655,37.645407,0.0,2022-12-07 03:28:29,15182867.0,1675.0,43197.0,0.0,0.0,0.0
289917,3193.0,108.988984,37.644059,0.0,2022-12-07 03:32:18,15183096.0,0.0,9.0,558.0,0.0,0.0


In [34]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-53.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-53.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 507714
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2966830951.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\2966830951.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 490358
490358
490358
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 399201
截断过长时间怠速的运行时间为：8459.83983039856s
开始线性插值...
插值后数据量 412518
插值运行时间为：3.8537144660949707s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,3194.0,108.988232,37.643544,0.0,2022-12-07 03:37:02,15183380.0,0.0,10.0,557.0,0.000000,0.000000
1,3194.0,108.988232,37.643544,0.0,2022-12-07 03:37:03,15183381.0,0.0,26.0,557.0,0.000000,0.000000
2,3194.0,108.988232,37.643544,0.0,2022-12-07 03:37:04,15183382.0,0.0,42.0,556.0,0.000000,0.000000
3,3194.0,108.988248,37.643547,0.0,2022-12-07 03:37:05,15183383.0,0.0,67.0,560.0,0.000000,0.000000
4,3194.0,108.988248,37.643552,0.0,2022-12-07 03:37:06,15183384.0,0.0,83.0,559.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
412513,3648.0,109.913376,38.569772,55.0,2022-12-31 23:58:59,17330297.0,94107.0,1008970.0,1256.0,15.277778,0.000000
412514,3648.0,109.913503,38.569840,55.0,2022-12-31 23:59:00,17330298.0,94122.0,1009223.0,1266.0,15.277778,0.000000
412515,3648.0,109.913655,38.569919,56.0,2022-12-31 23:59:01,17330299.0,94137.0,1009464.0,1268.0,15.555556,0.277778
412516,3648.0,109.913784,38.569987,56.0,2022-12-31 23:59:02,17330300.0,94154.0,1009713.0,1273.0,15.555556,0.000000


In [29]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-41.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-41.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 321419
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\4120121535.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\4120121535.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 306860
306860
306860
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 251307
截断过长时间怠速的运行时间为：3346.8214020729065s
开始线性插值...
插值后数据量 263204
插值运行时间为：2.505751371383667s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,1181.0,109.475112,38.111860,0.0,2022-10-08 00:26:53,9987971.0,0.0,15.0,548.0,0.000000,0.0
1,1181.0,109.475112,38.111860,0.0,2022-10-08 00:26:54,9987972.0,0.0,26.0,560.0,0.000000,0.0
2,1181.0,109.475112,38.111860,0.0,2022-10-08 00:26:55,9987973.0,0.0,43.0,559.0,0.000000,0.0
3,1181.0,109.475112,38.111855,0.0,2022-10-08 00:26:56,9987974.0,0.0,81.0,912.0,0.000000,0.0
4,1181.0,109.475112,38.111852,0.0,2022-10-08 00:26:57,9987975.0,0.0,173.0,770.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...
263199,1481.0,109.223416,38.242988,0.0,2022-10-18 17:25:17,10913075.0,442.0,4761.0,0.0,0.000000,0.0
263200,1482.0,109.223431,38.242999,6.0,2022-10-18 17:29:02,10913300.0,3.0,188.0,932.0,1.666667,0.0
263201,1483.0,109.225224,38.242908,4.0,2022-10-18 17:37:22,10913800.0,1.0,227.0,979.0,1.111111,0.0
263202,1484.0,109.227375,38.242903,6.0,2022-10-18 17:44:05,10914203.0,10.0,593.0,1180.0,1.666667,0.0


In [30]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-42.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-42.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 362334
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1529873191.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\1529873191.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 348485
348485
348485
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 296249
截断过长时间怠速的运行时间为：3532.077748298645s
开始线性插值...
插值后数据量 307959
插值运行时间为：2.881528854370117s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,1485.0,109.228488,38.242864,6.0,2022-10-18 17:49:41,10914539.0,4.0,251.0,766.0,1.666667,0.000000
1,1485.0,109.228504,38.242864,6.0,2022-10-18 17:49:42,10914540.0,6.0,266.0,611.0,1.666667,0.000000
2,1485.0,109.228528,38.242864,5.0,2022-10-18 17:49:43,10914541.0,7.0,282.0,556.0,1.388889,-0.277778
3,1485.0,109.228544,38.242864,5.0,2022-10-18 17:49:44,10914542.0,9.0,297.0,550.0,1.388889,0.000000
4,1485.0,109.228560,38.242864,5.0,2022-10-18 17:49:45,10914543.0,10.0,328.0,575.0,1.388889,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
307954,1948.0,109.835063,38.388515,0.0,2022-10-28 13:36:23,11763341.0,344.0,4284.0,330.0,0.000000,-0.000000
307955,1948.0,109.835063,38.388515,0.0,2022-10-28 13:36:24,11763342.0,344.0,4306.0,126.0,0.000000,0.000000
307956,1948.0,109.835063,38.388515,0.0,2022-10-28 13:36:25,11763343.0,344.0,4315.0,0.0,0.000000,0.000000
307957,1948.0,109.835063,38.388512,0.0,2022-10-28 13:36:26,11763344.0,344.0,4315.0,0.0,0.000000,0.000000


In [31]:
import datetime
import pandas as pd
import numpy as np
import xlsxwriter as xw
excel=pd.read_csv(r"D:\jupyter notebook 保存文件\还未预处理的\DL0320210200387-43.xlsx")
data = pd.DataFrame(excel)

#对于定位时间这一列，删除不是2022年的数据
data['定位年份']=data['定位时间'].apply(lambda r:r.split('-')[0])    
i = 0
while i <len(data):
    data.iloc[i, 18]=int(data.iloc[i,18])
    i += 1
print('转换完成')
data=data[data['定位年份']==2022]
print("删除时间不是2022年的之后的数据：",len(data))

#不要那个时间戳，自己定义时间戳
data['定位时间'] = pd.to_datetime(data['定位时间'], format= '%Y-%m-%d %H:%M:%S')
begin = datetime.datetime.strptime('2022-06-14 10:00:42', '%Y-%m-%d %H:%M:%S') 
data['时间戳'] = data['定位时间'].map(lambda x:(x-begin).total_seconds()) 
print("时间戳定义完成") 

#删除不需要的列，只保留'子行程id',‘经度’，‘纬度’，‘gps速度’，‘定位时间’，‘里程戳’，‘油量戳’，‘转数’，
data=data[['子行程id','经度','纬度','gps速度','定位时间','时间戳','里程戳','油量戳','转数']]
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
# 增加一列变换单位后的车速：索引为9
i=0
while i<len(data):
    #loc函数主要通过行标签索引行数据
    data.loc[i, 'gps车速m/s']=data.iat[i,3]/3.6
    i+=1
print("变换单位后的GPS车速，添加成功")

# 增加加速度列：索引为10
i=1
while i<len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        tf = data.iat[i,5]-data.iat[i-1,5]   #时间差
        sf = data.iat[i,9]-data.iat[i-1,9]   #速度差
        data.loc[i, '加速度(m/s^2)']=sf/tf
    else:
        data.loc[i, '加速度(m/s^2)'] = 0
    i+=1
data.loc[0, '加速度(m/s^2)'] = data.loc[1, '加速度(m/s^2)']  #补上空余值
print("加速度，添加成功")

####处理超速、加减速异常、有速度经纬度却不变、经纬度异常4种错误
#去除车速异常数据：车速大于120km/h，即每秒速度高于33m/s
#去除经度为0的数据
#去除纬度为0的数据
#去除经度和纬度不变化但GPS速度变化的数据
#去除加减速度异常，百公里加速时间不低于7秒，紧急刹车最大减速度处于7.5-8m/s^2
#先不对第一个和最后一个数据进行处理
i = 0
n = 0
while i<len(data)-1:
    
    #if data.loc[i,'GPS车速(m/s)'] > 33.3333 or data.loc[i,'经度'] == 0 or data.loc[i,'纬度'] == 0 or (data.loc[i,'经度'] - data.loc[i-1,'经度'] == 0 and data.loc[i,'维度'] - data.loc[i-1,'维度'] == 0 and data.loc[i,'GPS车速(m/s)'] != 0) or data.loc[i,'加速度(m/s^2)']>4 or data.loc[i,'加速度(m/s^2)']<-8:
    if data.iloc[i,9] > 33.3333 or data.iloc[i, 1] == 0 or data.iloc[i, 2] == 0 or (data.iloc[i, 1] - data.iloc[i-1,1] == 0 and data.iloc[i,2] - data.iloc[i-1,2] == 0 and data.iloc[i,9] != 0) or data.iloc[i,10]>4 or data.iloc[i,10]<-8:
        data.drop(i+n,inplace=True)  # drop靠“行名”进行查询，因此引入变量n
        n+=1
    else:
        i += 1
print("去除异常后的数据数量：",len(data))
#row_name = data._stat_axis.values.tolist()  #获取行名
#print(row_name)
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列
print(len(data))
#索引9数据类型改为浮点型
print(len(data))
i = 0
while i <len(data):
    data.iloc[i, 9]=float(data.iloc[i, 9])
    i += 1
print('转换完成')

'''#处理长期低速行驶，将其视为怠速状态
import time
#导入time模块
start=time.time()

print("开始处理长期低速行驶")
i = 0
lock = 0
count = 0
dele = []
while i < len(data):
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data[i][9]< 10/3.6:
            s1 = i
            lock = 1
        if lock == 1 and data[i][9] >= 10/3.6:
            # 只要出现速度大于10，就必须做决断。要么是存起来删除，要么是解锁，准备重新计时
            s2 = i
            # 低速时间超过50秒时，进行判断：
            # 如果速度为0的时间超过50，则不做处理
            # 如果速度为0的时间小于50，则认为是低速行驶，直接将此区间置0
            if data[s2][5] - data[s1][5] > 50:
                if count < 50:
                    low_speed = [s1+3,s2-3]  # 前后保留3个点的原始数值
                    dele.append(low_speed)
            lock = 0
            count = 0
        if lock == 1 and data[i][5] == 0:
            count += 1
        i += 1

    for i in range(len(dele)):
        for j in range(dele[i][0],dele[i][1]):
            data[j][1] = 0
    else:
        i += 1
print("处理完低速行驶后的数据量：",len(data))
end=time.time()
print("处理低速行驶的运行时间为：{}s".format(end-start))
'''



import time
#导入time模块
start=time.time()

#大于180秒的怠速时间处理成180秒
print("开始处理过长的怠速")
i = 0
n = 0
lock = 0
while i < len(data)-1:
    if data.iat[i,0]==data.iat[i-1,0]:
        if lock == 0 and data.iloc[i, 9] < (10 / 3.6): # 根据索引
            s = i
            #print(s)
            lock = 1
        if data.iloc[i, 9] < (10 / 3.6):
            count = (data.iloc[i,5] - data.iloc[s,5])
            if count > 180:
                data.drop(i+n, inplace=True) #根据行名
                n += 1
            else:
                i += 1
        else:
            lock = 0
            i += 1 
    else:
        i += 1
print("截断过长时间怠速的数据数量：",len(data))
arr = np.array(data)
data = pd.DataFrame(arr)        # 重新来回转变一次，将行名变成正常的序列

end=time.time()
print("截断过长时间怠速的运行时间为：{}s".format(end-start))


#线性插值
print("开始线性插值...")
import time
#导入time模块
start=time.time()
data_l= np.array(data) #先将数据框转换为数组
data_l = data_l.tolist()  #其次转换为列表

i = 1
while i <len(data_l):
    if data_l[i][0]==data_l[i-1][0]:
        t = data_l[i][5] - data_l[i-1][5]
        if t <= 6:    # 缺失点小于3秒时，进行插值
            for j in range(round(t-1)):   # 插入t-1个点
                diff = []
                new_p = []
                for m in range(11):  # 11为数据的维度
                    diff.append((data_l[i + j][m] - data_l[i-1][m]) / t)
                    new_p.append(data_l[i - 1][m] + diff[m] * (j + 1))
                data_l.insert(i+j,new_p)
        i+=1
    else:
        i += 1
print("插值后数据量",len(data_l))
end=time.time()
print("插值运行时间为：{}s".format(end-start))

data = pd.DataFrame(data_l,columns=['子行程id',"经度","纬度",'gps速度',"定位时间","时间戳",'里程戳','油量戳','转数',"gps车速m/s","加速度(m/s^2)"]
)
data.to_excel('预处理DL0320210200387-43.xlsx',index=None)
data

转换完成
删除时间不是2022年的之后的数据： 348756
时间戳定义完成
变换单位后的GPS车速，添加成功


C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\4263690035.py:42: RuntimeWarning: invalid value encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf
C:\Users\Administrator\AppData\Local\Temp\ipykernel_4596\4263690035.py:42: RuntimeWarning: divide by zero encountered in double_scalars
  data.loc[i, '加速度(m/s^2)']=sf/tf


加速度，添加成功
去除异常后的数据数量： 336548
336548
336548
转换完成
开始处理过长的怠速
截断过长时间怠速的数据数量： 285353
截断过长时间怠速的运行时间为：3384.857574701309s
开始线性插值...
插值后数据量 295312
插值运行时间为：2.5867562294006348s


,子行程id,经度,纬度,gps速度,定位时间,时间戳,里程戳,油量戳,转数,gps车速m/s,加速度(m/s^2)
0,1949.0,109.835072,38.388524,4.0,2022-10-28 13:38:58,11763496.0,2.0,178.0,751.0,1.111111,0.277778
1,1949.0,109.835088,38.388528,5.0,2022-10-28 13:38:59,11763497.0,4.0,251.0,1029.0,1.388889,0.277778
2,1949.0,109.835104,38.388532,6.0,2022-10-28 13:39:00,11763498.0,5.0,326.0,1153.0,1.666667,0.277778
3,1949.0,109.835119,38.388528,5.0,2022-10-28 13:39:01,11763499.0,7.0,338.0,692.0,1.388889,-0.277778
4,1949.0,109.835175,38.388504,3.0,2022-10-28 13:39:02,11763500.0,9.0,371.0,726.0,0.833333,-0.555556
...,...,...,...,...,...,...,...,...,...,...,...
295307,2399.0,108.941536,37.652608,0.0,2022-11-05 23:13:49,12489187.0,561.0,9978.0,141.0,0.000000,0.000000
295308,2399.0,108.941536,37.652612,0.0,2022-11-05 23:13:50,12489188.0,561.0,9985.0,0.0,0.000000,0.000000
295309,2399.0,108.941536,37.652612,0.0,2022-11-05 23:13:51,12489189.0,561.0,9985.0,0.0,0.000000,0.000000
295310,2399.0,108.941536,37.652612,0.0,2022-11-05 23:13:52,12489190.0,561.0,9985.0,0.0,0.000000,0.000000
